In [ ]:
from datetime import date
import os

from vytallink_health_kit.application.use_cases import (
    BuildReadinessReportInput,
    BuildReadinessReportUseCase,
)
from vytallink_health_kit.infrastructure.settings import load_vytallink_settings
from vytallink_health_kit.infrastructure.vytallink_client import VytalLinkRESTClient

settings = load_vytallink_settings()
print({
    "base_url": settings.base_url,
    "api_mode": settings.api_mode,
    "sleep_path": settings.sleep_path,
    "heart_rate_path": settings.heart_rate_path,
    "activity_path": settings.activity_path,
    "direct_login_path": settings.direct_login_path,
    "metrics_path": settings.metrics_path,
    "sleep_value_type": settings.sleep_value_type,
    "heart_rate_value_type": settings.heart_rate_value_type,
    "activity_value_type": settings.activity_value_type,
    "has_word": bool(settings.word or os.getenv("VYTALLINK_WORD")),
    "has_code": bool(settings.code or os.getenv("VYTALLINK_CODE")),
})

In [ ]:
from uuid import uuid4

import httpx

conversation_id = f"notebook-{uuid4()}"
base_url = str(settings.base_url).rstrip("/")

client = httpx.Client(base_url=base_url, timeout=settings.timeout_seconds)
headers = {"openai-conversation-id": conversation_id}
credentials = {
    "word": settings.word or os.getenv("VYTALLINK_WORD", ""),
    "code": settings.code or os.getenv("VYTALLINK_CODE", ""),
}

login_response = client.post("/login", data=credentials)
direct_login_response = client.post(
    settings.direct_login_path,
    data=credentials,
    headers=headers,
 )

print({
    "conversation_id": conversation_id,
    "login_status": login_response.status_code,
    "login_error_hint": "Invalid word or code" in login_response.text,
    "direct_login_status": direct_login_response.status_code,
    "direct_login_json": direct_login_response.json(),
})

In [ ]:
provider = VytalLinkRESTClient(settings=settings)
use_case = BuildReadinessReportUseCase(health_data_provider=provider)

report = use_case.execute(
    BuildReadinessReportInput(
        end_date=date.today(),
        days=7,
        include_narrative=False,
    )
)

report.readiness.model_dump()

In [ ]:
print(report.markdown)